# Personalized Lorlatinib Dosing 

**Purpose:** Demonstrate that patients with different c_in values (CYP3A4 metabolizer phenotypes) can achieve similar PK profiles to the reference (c_in=1.0, 100mg QD) by adjusting the dosing regimen.

**Author:** Julia Pelesko   
**Last Updated:** July 2026 

---

## Overview

This notebook demonstrates three personalized dosing strategies:

1. **Dose Adjustment:** Modulate dose while keeping interval constant to minimize difference between AUC of average metabolizer (c_in=1.0, 100mg QD) and of patient 
2. **Frequency Adjustment:** Modulate dosing interval while keeping dose constant to minimize difference between C_avg of average metabolizer (c_in=1.0, 100mg QD) and of patient 
3. **Joint Adjustment:** Modulate both frequency and dose together to minimize difference between C_avg, C$_{max}$, and C$_{min}$ of average metabolizer (c_in=1.0, 100mg QD) and of patient

The components of the simulation are:

1. **Section 1:** Import packages.
2. **Section 2:** Define Hybrid PK model. This is used to initialize a virual patient and simulate dosing. The class will return drug concentration over time in each compartment (depot, central, and peripheral) and clearance over time. 
3. **Section 3:** Define helper functions. The calculate_final_metrics function calculates PK metrics at last dosing interval, even if simulation is cut off before dosing interval is complete. The find_steady_state function returns the number of doses required to reach steady state for a given c_in. The find_optimal_dose function uses L-BFGS-B optimization to find the dose that minimizes the difference between AUC of average metabolizer (c_in=1.0, 100mg QD) and of the simulated patient. The find_optimal_interval uses the brentq optimizer to find the dosing interval that minimizes the difference between C_avg of average metabolizer (c_in=1.0, 100mg QD) and of the simulated patient. The find_optimal_dose_and_interval find the optimal dose, frequency pair that minimizes the difference between C_avg, C$_{max}$, and C$_{min}$ of average metabolizer (c_in=1.0, 100mg QD) and of teh simulated patient.
4. **Section 4:** Reference Patient Simulation. Initialze average metabolizer (c_in=1.0, 100mg QD), simulate 24 days of SOC dosing, calculate final metrics. Visualize plasma concentration v time for first 7 days of dosing for 3 different metabolizers (c_in=0.4, c_in=1.0, c_in=2.5).
5. **Section 5:** Patient class. This is used to initialize a virual patient, run all 3 dose regimen optimizations, simulate dosing for SOC and all 3 optimal regimens, and return PK metrics for all simulations.
6. **Section 6:** Visualization. Visualize plasma concentration v time curve and bar graph of PK metrics for any simulation.
7. **Section 7:** Analysis. Define a slow and fast metabolizer. Find all optimal dosing regimens, simulate dosing regimens, return table of Pk metric comparison, visualize dosing regimens.
8. **Section 8:** PK Metrics vs c_in. Sweep c_in from 0.05 to 10 and simulate SOC dosing. Plot PK metrics vs c_in.
9. **Section 9:** Solution Set Visualization. Plot PK Metrics vs Dose for 3 metabolizer phenotypes. Plot PK Metrics vs Dosing Frequency for 3 metabolizer phenotypes. 

 

## 1. Setup and Imports

In [ ]:
import numpy as np
from scipy.integrate import odeint
from scipy.optimize import minimize
from scipy.optimize import brentq
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

# Global plot formatting
import matplotlib
matplotlib.rcParams.update({
    'font.family': 'Arial',
    'font.size': 18,
})

## 2. PK Model Implementation

The Hybrid Model combines:
- Chen et al. (2021) 2-compartment structure with novel c_in personalization parameter representing individual CYP3A4 activity

### Model Equation:

$$CL(t, c_{in}) = c_{in} \times [CL_{initial} + (CL_{max} - CL_{initial}) \times (1 - e^{-t/\tau})]$$

In [ ]:
class HybridModel:
    """
    Hybrid model: Chen et al. structure + c_in personalization.

    CL(t, c_in) = c_in × [CL_initial + (CL_max - CL_initial) × (1 - exp(-t/τ))]
    """

    def __init__(self, c_in=1.0):
        # Parameters set to population average values reported in Chen et al. (2021) 
        self.ka = 3.113          # h^-1, absorption rate constant
        self.CL_initial = 9.035  # L/h, initial clearance
        self.CL_max = 14.472     # L/h, steady-state clearance
        self.tau = 50.251        # h, induction time constant (= 1/IND = 1/0.0199)
        self.V2 = 120.511        # L, central volume of distribution
        self.V3 = 154.905        # L, peripheral volume of distribution
        self.Q = 22.002          # L/h, inter-compartmental clearance
        self.F = 0.759           # bioavailability

        self.c_in = c_in

    def clearance(self, t):
        """Time-varying clearance personalized by c_in."""
        induction_factor = 1 - np.exp(-t / self.tau)
        CL_t = self.CL_initial + (self.CL_max - self.CL_initial) * induction_factor
        return self.c_in * CL_t

    def ode_system(self, y, t):
        """
        Right-hand side of the 3-compartment ODE system (depot, central, peripheral).

        Computes the time derivatives of drug amount in each compartment from
        first-order absorption (ka), time-varying central elimination (CL(t)/V2),
        and inter-compartmental distribution (Q). Intended for scipy.integrate.odeint.

        Parameters
        ----------
        y : list  — current amounts [A_depot, A_central, A_peripheral]
        t : float — current time in hours (used to evaluate the time-varying CL)

        Returns
        -------
        derivatives : list — [dA_depot, dA_central, dA_peripheral]
        """
        A_depot, A_central, A_peripheral = y
        CL_t = self.clearance(t)

        dA_depot = -self.ka * A_depot
        dA_central = (self.ka * A_depot
                     - (CL_t / self.V2) * A_central
                     - (self.Q / self.V2) * A_central
                     + (self.Q / self.V3) * A_peripheral)
        dA_peripheral = (self.Q / self.V2) * A_central - (self.Q / self.V3) * A_peripheral

        return [dA_depot, dA_central, dA_peripheral]

    def simulate_multiple_doses(self, dose_mg=100.0, n_doses=15, dosing_interval=24.0, n_points=5000):
        """Simulate multiple dose PK profile."""
        duration_hours = n_doses * dosing_interval + 24
        dose_ug = dose_mg * 1000 * self.F

        t = np.linspace(0, duration_hours, n_points)

        y0 = [0.0, 0.0, 0.0]
        dose_times = [i * dosing_interval for i in range(n_doses)]

        t_segments = []
        solution_segments = []
        current_state = y0
        t_start = 0

        for dose_idx in range(n_doses):
            if dose_idx < n_doses - 1:
                t_end = dose_times[dose_idx + 1]
            else:
                t_end = duration_hours

            t_segment = t[(t >= t_start) & (t < t_end)]
            if len(t_segment) == 0:
                continue

            current_state[0] += dose_ug

            solution_segment = odeint(self.ode_system, current_state, t_segment)
            t_segments.extend(t_segment)
            solution_segments.append(solution_segment)

            if len(solution_segment) > 0:
                current_state = solution_segment[-1, :]
            t_start = t_end

        solution_full = np.vstack(solution_segments)
        A_central = solution_full[:, 1]
        C_central = (A_central / self.V2) 

        return {'time': np.array(t_segments), 'C_central': C_central, 'dose_times': dose_times}

print("✓ HybridModel class defined")

## 3. Helper Functions

In [ ]:
def calculate_final_metrics(simulation, dosing_interval=24.0):
    """Calculate PK metrics at last dosing interval."""
    t = simulation['time']
    C = simulation['C_central']

    #Use the actual last dose time to define the final interval
    if 'dose_times' in simulation:
        t_last_dose = simulation['dose_times'][-1]
    else:
        #Fallback if dose_times not available
        t_last_dose = t[-1] - dosing_interval - 24

    mask = (t >= t_last_dose) & (t < t_last_dose + dosing_interval)

    t_f = t[mask]
    C_f = C[mask]

    Cmax_f = np.max(C_f)
    Cmin_f = np.min(C_f)
    AUC_f = np.trapezoid(C_f, t_f - t_f[0])  

    return {
        'Cmax_f': Cmax_f,
        'Cmin_f': Cmin_f,
        'AUC_f': AUC_f,
        'Cavg_f': AUC_f / dosing_interval
    }

print("\u2713 calculate_final_metrics defined")


In [ ]:
def find_steady_state(c_in, dose_mg=100.0, dosing_interval=24.0,
                       tol=0.01, min_doses=10, max_doses=300):
    """
    Return the number of doses required to reach steady state for a given c_in.

    Simulates repeated dosing in blocks of 5 until the AUC of the last
    interval changes by less than tol relative to the previous check.
    Because CL(t) does not depend on dose, any reference dose_mg can be used.

    Parameters
    ----------
    c_in          : float — CYP3A4 activity parameter
    dose_mg       : float — reference dose in mg (default 100 mg)
    dosing_interval : float — dosing interval in hours (default 24 h)
    tol           : float — relative AUC convergence threshold (default 1 %)
    min_doses     : int   — minimum doses before convergence is checked
    max_doses     : int   — hard cap to prevent runaway loops

    Returns
    -------
    n_doses : int — number of doses at which steady state is declared
    """
    model = HybridModel(c_in=c_in)
    prev_auc = None

    for n in range(min_doses, max_doses + 1, 5):
        sim = model.simulate_multiple_doses(
            dose_mg=dose_mg, n_doses=n, dosing_interval=dosing_interval
        )
        auc = calculate_final_metrics(sim, dosing_interval)['AUC_f']

        if prev_auc is not None and abs(auc - prev_auc) / prev_auc < tol:
            return n

        prev_auc = auc

    return max_doses  # fallback if convergence not reached

print('\u2713 find_steady_state defined')


In [ ]:
def find_optimal_dose(c_in_target, target_metrics, dosing_interval=24.0):
    """
    Find optimal dose for a given c_in to match AUC of average metabolizer at 100mgQD.

    Parameters:
    -----------
    c_in_target : float
        The c_in value for this patient
    target_metrics : dict
        Target PK metrics to match (from reference c_in=1.0)
    dosing_interval : float
        Dosing interval in hours

    Returns:
    --------
    optimal_dose : float
        Dose in mg that matches target AUC
    """
    n_doses = find_steady_state(c_in_target, dosing_interval=dosing_interval)

    # Scale initial guess and upper bound with c_in so the optimizer starts
    # close to the expected optimum (optimal dose ∝ c_in in linear PK).
    dose_init  = 100.0 * c_in_target
    dose_upper = max(600.0, 200.0 * c_in_target)

    def objective(dose):
        model = HybridModel(c_in=c_in_target)
        sim = model.simulate_multiple_doses(
            dose_mg=dose[0], n_doses=n_doses, dosing_interval=dosing_interval
        )
        metrics = calculate_final_metrics(sim, dosing_interval)
        error = ((metrics['AUC_f'] - target_metrics['AUC_f']) / target_metrics['AUC_f'])**2
        return error

    result = minimize(objective, x0=[dose_init],
                      bounds=[(10.0, dose_upper)], method='L-BFGS-B')
    return result.x[0]

print('\u2713 find_optimal_dose defined')


In [ ]:
def find_optimal_interval(c_in_target, target_metrics, dose_mg=100.0):
    """
    Find optimal dosing interval for a given c_in to match target Cavg.

    At steady state, AUC per interval = F*dose/CL (independent of tau), so
    Cavg = AUC/tau is strictly monotone decreasing. brentq finds the unique
    root of Cavg(tau) - target_Cavg = 0, which is more robust than a
    gradient-based optimizer on this inherently 1-D monotone problem.

    Parameters:
    -----------
    c_in_target : float
    target_metrics : dict
    dose_mg : float — fixed dose in mg

    Returns:
    --------
    optimal_interval : float — dosing interval in hours
    """
    target_cavg = target_metrics['Cavg_f']

    def cavg_at_tau(tau):
        # Enough doses to cover ~500 h so induction
        # (tau_ind = 50.25 h) is complete at every candidate interval.
        n_doses = max(20, int(500 / tau))
        model = HybridModel(c_in=c_in_target)
        sim = model.simulate_multiple_doses(
            dose_mg=dose_mg, n_doses=n_doses, dosing_interval=tau
        )
        return calculate_final_metrics(sim, dosing_interval=tau)['Cavg_f']

    tau_min, tau_max = 1.0, 96.0

    cavg_lo = cavg_at_tau(tau_min)   
    cavg_hi = cavg_at_tau(tau_max)   

    # If target falls outside the achievable range, return the nearest bound.
    if cavg_lo < target_cavg:
        return tau_min
    if cavg_hi > target_cavg:
        return tau_max

    optimal_tau = brentq(
        lambda tau: cavg_at_tau(tau) - target_cavg,
        tau_min, tau_max,
        xtol=0.05   # converge to within 3 minutes — sufficient for dosing
    )
    return optimal_tau

print('\u2713 find_optimal_interval defined')


In [ ]:
def find_optimal_dose_and_interval(c_in_target, target_metrics,
                                   weights=(1.0, 1.0, 1.0),
                                   cmin_lower_bound=None):
    """
    Find optimal dose AND dosing interval simultaneously for a given c_in.

    Minimizes weighted sum of squared relative errors for Cmax, Cmin, and Cavg.
    A 3x3 grid of initial guesses is used to reduce sensitivity to local minima.

    Parameters
    ----------
    c_in_target : float
    target_metrics : dict
    weights : tuple (w_auc, w_cmax, w_cmin)
    cmin_lower_bound : float or None
        If set, enforces Cmin >= cmin_lower_bound (ng/mL) via a quadratic
        penalty term. Violations are penalised as 1e3 * ((lb - Cmin)/lb)^2,
        which dominates the objective whenever Cmin falls below the threshold.

    Returns
    -------
    (optimal_dose, optimal_interval) : (float, float)
    """
    n_doses = find_steady_state(c_in_target, dose_mg=100.0, dosing_interval=24.0)

    w_cavg, w_cmax, w_cmin = weights

    def objective(params):
        dose, interval = params[0], params[1]

        model = HybridModel(c_in=c_in_target)
        sim = model.simulate_multiple_doses(
            dose_mg=dose, n_doses=n_doses, dosing_interval=interval
        )
        metrics = calculate_final_metrics(sim, dosing_interval=interval)

        err_cavg = w_cavg * ((metrics['Cavg_f'] - target_metrics['Cavg_f'])
                              / target_metrics['Cavg_f'])**2
        err_cmax = w_cmax * ((metrics['Cmax_f'] - target_metrics['Cmax_f'])
                              / target_metrics['Cmax_f'])**2
        err_cmin = w_cmin * ((metrics['Cmin_f'] - target_metrics['Cmin_f'])
                              / target_metrics['Cmin_f'])**2

        # Optional quadratic penalty if Cmin falls below the lower bound
        penalty = 0.0
        if cmin_lower_bound is not None and metrics['Cmin_f'] < cmin_lower_bound:
            penalty = 1e3 * ((cmin_lower_bound - metrics['Cmin_f'])
                             / cmin_lower_bound) ** 2

        return err_cavg + err_cmax + err_cmin + penalty

    # Scale dose grid and upper bound with c_in (optimal dose ∝ c_in in linear PK)
    dose_upper = max(600.0, 200.0 * c_in_target)

    # 3x3 grid of initial guesses for robustness against local minima
    best_result = None
    best_error  = np.inf

    for dose_init in [50.0 * c_in_target, 100.0 * c_in_target, 200.0 * c_in_target]:
        for interval_init in [12.0, 24.0, 36.0]:
            result = minimize(
                objective,
                x0=[dose_init, interval_init],
                bounds=[(10.0, dose_upper), (6.0, 48.0)],
                method='L-BFGS-B'
            )
            if result.fun < best_error:
                best_error  = result.fun
                best_result = result

    return best_result.x[0], best_result.x[1]

print('\u2713 find_optimal_dose_and_interval defined')


## 4. Reference Patient Simulation

Establish baseline PK for average metabolizer (c_in = 1.0, 100mg QD)

In [ ]:
print("="*80)
print("PERSONALIZED DOSING DEMONSTRATION")
print("Hybrid Model (Chen + c_in)")
print("="*80)
print()
print("Calculating personalized dosing strategies...")
print("="*80)

# Reference: Average metabolizer (c_in = 1.0, 100mg QD)
print("\n1. Reference Patient (Average Metabolizer):")
print("   c_in = 1.0, Dose = 100 mg QD")
model_ref = HybridModel(c_in=1.0)
sim_ref = model_ref.simulate_multiple_doses(dose_mg=100, n_doses=24, dosing_interval=24)
metrics_ref = calculate_final_metrics(sim_ref, dosing_interval=24)

print(f"   Cmax_ss = {metrics_ref['Cmax_f']:.1f} ng/mL")
print(f"   Cmin_ss = {metrics_ref['Cmin_f']:.1f} ng/mL")
print(f"   AUC_ss  = {metrics_ref['AUC_f']:.1f} ng·h/mL")
print(f"   Cavg_ss = {metrics_ref['Cavg_f']:.1f} ng/mL")

In [ ]:
#Visualize Conc v Time for 3 Different Metabolizers 
PHENO_3 = [
    (0.4, 'c_in=0.4',   dict(color='gold',       linewidth=4, alpha=0.9)),
    (1.00, 'c_in=1.0', dict(color='black', linewidth=4, alpha=1.0)),
    (2.5, 'c_in=2.5',   dict(color='darkred',    linewidth=4, alpha=0.9)),
]

fig_multi, ax_multi = plt.subplots(figsize=(10, 5))

for c_in_val, label, kwargs in PHENO_3:
    sim    = HybridModel(c_in=c_in_val).simulate_multiple_doses(dose_mg=100, n_doses=7, dosing_interval=24)
    t_days = sim['time'] / 24
    ax_multi.plot(t_days, sim['C_central'], label=label, **kwargs)

ax_multi.set_xlabel('Time (days)', fontsize=18)
ax_multi.set_ylabel('lorlatinib (ng/mL)', fontsize=18)
ax_multi.set_xlim(0, 7)
ax_multi.set_xticks([0, 1, 2, 3, 4, 5, 6, 7])
ax_multi.spines['top'].set_visible(False)
ax_multi.spines['right'].set_visible(False)

handles, labels = ax_multi.get_legend_handles_labels()
ax_multi.legend(handles, labels, loc='best', bbox_to_anchor=(1.05, 0.95),
                borderaxespad=0, frameon=False)

save_path = '/Users/80031987/Desktop/Basanta_Marusyk_Lab_2026/Lorlatinib_Human_PKProject_30Jan2025/Hybrid_Model/Figures/LorConcvTime.pdf'
#plt.savefig('/Users/80031987/Desktop/LorConcTime.pdf', dpi=300, bbox_inches='tight', transparent=True)

plt.tight_layout()
plt.show()

## 5. Patient Class

`Patient` encapsulates all four dosing strategies for a given CYP3A4 phenotype.
Calling `run_all(target_metrics)` runs the unadjusted baseline and all three
optimizers; results are stored as `Regimen` objects on the patient instance.

In [ ]:
from dataclasses import dataclass

@dataclass
class Regimen:
    """One (dose, interval) pair together with its simulation and PK metrics."""
    dose_mg:    float
    interval_h: float
    sim:        dict
    metrics:    dict


class Patient:
    """
    All dosing strategies for a single CYP3A4 phenotype.

    Parameters
    ----------
    c_in : float — CYP3A4 activity parameter

    Attributes set by run_all()
    ---------------------------
    unadj     : Regimen — 100 mg Q24h baseline
    dose_opt  : Regimen — dose-only optimization (Q24h)
    freq_opt  : Regimen — frequency-only optimization (100 mg)
    joint_opt : Regimen — joint dose + interval optimization
    """

    def __init__(self, c_in):
        self.c_in  = c_in
        self.unadj = None
        self.dose_opt    = None
        self.freq_opt    = None
        self.joint_opt   = None

    # ── Internal ──────────────────────────────────────────────────────────────
    def _simulate(self, dose_mg, interval_h, n_doses=None, min_days=18):
        """Simulate to steady state and return a Regimen."""
        if n_doses is None:
            n_doses = find_steady_state(self.c_in, dosing_interval=interval_h)
        # Ensure the simulation covers at least min_days regardless of when
        # steady state is declared (short intervals can fall short of xlim)
        import math
        n_doses = max(n_doses, math.ceil(min_days * 24 / interval_h))
        sim     = HybridModel(c_in=self.c_in).simulate_multiple_doses(
            dose_mg=dose_mg, n_doses=n_doses, dosing_interval=interval_h
        )
        metrics = calculate_final_metrics(sim, dosing_interval=interval_h)
        return Regimen(dose_mg=dose_mg, interval_h=interval_h, sim=sim, metrics=metrics)

    def set_regimen(self, dose_mg, interval_h):
        """
        Simulate a manually specified regimen and store it as self.custom.
        Can be called independently of run_all().

        Parameters
        ----------
        dose_mg    : float — dose in mg
        interval_h : float — dosing interval in hours
        """
        self.custom = self._simulate(dose_mg, interval_h)
        return self

    # ── Public ────────────────────────────────────────────────────────────────
    def run_all(self, target_metrics, weights=(1.0, 1.0, 1.0), cmin_lb=None):
        """
        Run unadjusted baseline + all three optimization strategies.
        Returns self to allow chaining: p = Patient(...).run_all(metrics_ref)

        Parameters
        ----------
        target_metrics : dict        — reference PK metrics to match
        weights        : tuple       — (w_cavg, w_cmax, w_cmin) for joint optimizer
        cmin_lb        : float|None  — Cmin lower-bound penalty for joint optimizer
        """
        print(f"c_in = {self.c_in}  (9 joint-optimizer starts — may take a moment)")

        # Unadjusted: 100 mg Q24h
        self.unadj = self._simulate(100.0, 24.0)

        # Dose optimization: optimal dose, Q24h fixed
        dose = find_optimal_dose(self.c_in, target_metrics, dosing_interval=24.0)
        self.dose_opt = self._simulate(dose, 24.0)

        # Frequency optimization: 100 mg fixed, optimal interval
        interval = find_optimal_interval(self.c_in, target_metrics, dose_mg=100.0)
        self.freq_opt = self._simulate(100.0, interval)

        # Joint optimization: both dose and interval free
        dose_j, interval_j = find_optimal_dose_and_interval(
            self.c_in, target_metrics, weights=weights, cmin_lower_bound=cmin_lb
        )
        self.joint_opt = self._simulate(dose_j, interval_j)

        # Summary
        w = 14
        print(f"  {'Strategy':<{w}} {'Dose (mg)':>10} {'Interval (h)':>14}")
        print(f"  {'-'*(w+26)}")
        for label, r in [('Unadjusted',  self.unadj),
                          ('Dose-opt',    self.dose_opt),
                          ('Freq-opt',    self.freq_opt),
                          ('Joint-opt',   self.joint_opt)]:
            print(f"  {label:<{w}} {r.dose_mg:>10.1f} {r.interval_h:>14.1f}")

        return self


print("\u2713 Regimen and Patient defined")


## 6. Visualization

`plot_pk_curves` and `plot_metrics_bars` both accept a `strategies` list so you
can show any subset of `['unadj', 'dose_opt', 'freq_opt', 'joint_opt']` on the
same axes 

In [ ]:
# -- Strategy display configuration ------------------------------------------
_STRAT = {
    'unadj':     {'ls': '-',   'lw': 2,   'alpha': 0.7,
                  'label': lambda p: f'c$_{{in}}$={p.c_in}, 100mg QD'},
    'dose_opt':  {'ls': '-',   'lw': 2,   'alpha': 0.7,
                  'label': lambda p: f'c$_{{in}}$={p.c_in}, {p.dose_opt.dose_mg:.0f}mg QD'},
    'freq_opt':  {'ls': '-',  'lw': 2,   'alpha': 0.7,
                  'label': lambda p: f'c$_{{in}}$={p.c_in}, 100mg Q{p.freq_opt.interval_h:.1f}h'},
    'joint_opt': {'ls': '-',   'lw': 2.5, 'alpha': 1.0,
                  'label': lambda p: (f'c$_{{in}}$={p.c_in}, ' +
                                      f'{p.joint_opt.dose_mg:.0f}mg ' +
                                      f'Q{p.joint_opt.interval_h:.1f}h')},
    'custom':    {'ls': '-',   'lw': 2,   'alpha': 0.9,
                  'label': lambda p: (f'c$_{{in}}$={p.c_in}, ' +
                                      f'{p.custom.dose_mg:.0f}mg ' +
                                      f'Q{p.custom.interval_h:.1f}h')},
}

# -- Per-phenotype colour palettes --------------------------------------------
COLORS_SLOW = {   # c_in < 1
    'unadj':     'gold',
    'dose_opt':  'g',
    'freq_opt':  'g',
    'joint_opt': 'g',
    'custom':    'g',
}

COLORS_FAST = {   # c_in > 1
    'unadj':     'darkred', 
    'dose_opt':  'blue',
    'freq_opt':  'blue',
    'joint_opt': 'blue',
    'custom':    'blue',
}

def _get_colors(patient):
    """Return the appropriate colour dict based on patient.c_in."""
    return COLORS_SLOW if patient.c_in < 1 else COLORS_FAST


def _add_xaxis_break(ax, d=0.018):
    """
    Draw // break marks at the left edge of the x-axis to indicate elapsed time.
    Uses axes-fraction coordinates so the marks are size-invariant.
    """
    kw = dict(transform=ax.transAxes, color='k', clip_on=False,
              lw=1.5, solid_capstyle='round', zorder=10)
    for x0 in [0.0, 0.022]:          # two parallel slashes
        ax.plot([x0 - d, x0 + d], [-1.6*d, 1.6*d], **kw)
    from matplotlib.patches import FancyBboxPatch
    ax.add_patch(FancyBboxPatch(
        (-0.005, -0.04), 0.055, 0.08,
        boxstyle='square,pad=0', transform=ax.transAxes,
        facecolor='white', edgecolor='none', clip_on=False, zorder=9
    ))
    for x0 in [0.0, 0.022]:
        ax.plot([x0 - d, x0 + d], [-1.6*d, 1.6*d], **kw)


def plot_pk_curves(patient, sim_ref, ax, strategies, xlim, ylim):
    """
    Plot lorlatinib concentration-vs-time curves for a patient's dosing strategies.

    Draws the reference (c_in=1, 100 mg QD) profile in black plus one curve per
    requested strategy (coloured by phenotype via _get_colors), restricted to the
    display window, then styles the axis and adds an x-axis break mark.

    Parameters
    ----------
    patient    : Patient — patient whose regimens are plotted
    sim_ref    : dict — reference simulation (time, C_central) for c_in=1, 100 mg QD
    ax         : matplotlib Axes — axis to draw on
    strategies : list — strategy keys to plot (e.g. 'unadj', 'dose_opt', ...)
    xlim       : tuple — (min, max) x-axis limits in days
    ylim       : tuple — (min, max) y-axis limits in ng/mL

    Returns
    -------
    None — draws on ax in place
    """
    colors = _get_colors(patient)
    t0 = xlim[0]

    t_ref  = sim_ref['time'] / 24
    mask   = t_ref >= t0
    ax.plot(t_ref[mask], sim_ref['C_central'][mask],
            'black', lw=2, label='c$_{in}$=1, 100mg QD')

    for strat in strategies:
        cfg  = _STRAT[strat]
        r    = getattr(patient, strat)
        t    = r.sim['time'] / 24
        mask = t >= t0
        ax.plot(t[mask], r.sim['C_central'][mask],
                color=colors[strat], ls=cfg['ls'], lw=cfg['lw'], alpha=cfg['alpha'],
                label=cfg['label'](patient))

    x_pad = 0.09 * (xlim[1] - xlim[0])
    ax.set_xlim(xlim[0] - x_pad, xlim[1]); ax.set_ylim(*ylim)
    ax.set_xlabel('Time (days)', fontsize=18)
    ax.set_xticks([16, 17, 18])
    ax.set_ylabel('lorlatinib (ng/mL)', fontsize=18)
    ax.legend(loc='upper left', bbox_to_anchor=(1.0, 1.0), fontsize=18, frameon=False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    _add_xaxis_break(ax)


def plot_metrics_bars(patient, metrics_ref, ax, strategies,
                      metric_keys=('Cavg_f', 'Cmax_f', 'Cmin_f'),
                      x_labels=('AUC/$\\tau$', 'C$_{max}$', 'C$_{min}$')):
    """
    Plot PK metrics for each strategy as a bar chart normalised to SOC (% of reference).

    For each metric in metric_keys, draws the reference at 100% plus one bar per
    strategy (its value as a percentage of the reference), with a shaded +/-10%
    target band. Bars are coloured by phenotype via _get_colors.

    Parameters
    ----------
    patient     : Patient — patient whose regimen metrics are plotted
    metrics_ref : dict — reference PK metrics (c_in=1, 100 mg QD) used as 100%
    ax          : matplotlib Axes — axis to draw on
    strategies  : list — strategy keys to plot
    metric_keys : tuple — metric dict keys to display (default Cavg_f, Cmax_f, Cmin_f)
    x_labels    : tuple — x-axis tick labels for the three metrics

    Returns
    -------
    None — draws on ax in place
    """
    colors = _get_colors(patient)
    n      = 1 + len(strategies)
    w      = 0.7 / n
    x      = np.arange(3)
    offs   = np.linspace(-(n-1)/2, (n-1)/2, n) * w
    ref_v  = [metrics_ref[k] for k in metric_keys]

    alphas = {'unadj': 0.9, 'dose_opt': 0.9, 'freq_opt': 0.6, 'joint_opt': 0.7, 'custom': 0.7}

    ax.bar(x + offs[0], [100]*3, w,
           label='c$_{in}$=1, 100mg QD', color='black') #'darkorange'

    for i, strat in enumerate(strategies):
        r    = getattr(patient, strat)
        norm = [(r.metrics[k] / rv) * 100 for k, rv in zip(metric_keys, ref_v)]
        ax.bar(x + offs[i+1], norm, w,
               label=_STRAT[strat]['label'](patient),
               color=colors[strat], alpha=alphas[strat])

    ax.axhline(100, color='black', ls='-',  lw=1.5)
    ax.axhline(90,  color='gray',  ls='--', lw=1, alpha=0.5)
    ax.axhline(110, color='gray',  ls='--', lw=1, alpha=0.5)
    ax.axhspan(90, 110, alpha=0.15, color='green', label='±10% target')
    ax.set_xticks(x); ax.set_xticklabels(x_labels, fontsize=18)
    ax.set_ylabel('% of SOC', fontsize=18)
    ax.legend(loc='upper left', bbox_to_anchor=(1.0, 1.0), fontsize=18, frameon=False)
    all_v = [100]*3 + [(getattr(patient, s).metrics[k] / rv) * 100
                       for s in strategies
                       for k, rv in zip(metric_keys, ref_v)]
    ax.set_ylim(min(all_v) * 0.85, max(all_v) * 1.15)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)


print('✓ plot_pk_curves and plot_metrics_bars defined')

## 7. Analysis

Create a `Patient`, call `run_all`, then pass it to the plot functions.
Add one cell per c_in value of interest.

In [ ]:
ALL_STRATS = ['unadj', 'dose_opt', 'freq_opt', 'joint_opt', 'custom']

# Create a PK metrics comparison table for a patient
def print_metrics(patient, metrics_ref, strategies=None):
    """
    Print a formatted table comparing PK metrics across strategies.

    Tabulates Cmax, Cmin, Cavg, and AUC for the reference metabolizer alongside
    each requested strategy's regimen metrics. If strategies is None, all
    strategies in ALL_STRATS are shown.

    Parameters
    ----------
    patient     : Patient — patient whose regimen metrics are tabulated
    metrics_ref : dict — reference PK metrics (shown as the 'Reference' column)
    strategies  : list or None — strategy keys to include (default ALL_STRATS)

    Returns
    -------
    None — prints the table to stdout
    """
    if strategies is None:
        strategies = ALL_STRATS
    keys   = [('Cmax_f', 'Cmax (ng/mL)'), ('Cmin_f', 'Cmin (ng/mL)'),
              ('Cavg_f', 'Cavg (ng/mL)'), ('AUC_f',  'AUC (ng\u00b7h/mL)')]
    labels = ['Reference'] + [s for s in strategies]
    vals   = [metrics_ref] + [getattr(patient, s).metrics for s in strategies]
    w = 16
    header = f"  {'Metric':<{w}}" + "".join(f"{l:>14}" for l in labels)
    print(header)
    print("  " + "-" * (w + 14 * len(labels)))
    for key, label in keys:
        row = f"  {label:<{w}}" + "".join(f"{v[key]:>14.1f}" for v in vals)
        print(row)

print("\u2713 Helpers ready — create patients below")


In [ ]:
# ── Fast metabolizer ──────────────────────────────────────────────────────────
c_in_fast = 2.5
fast = Patient(c_in=c_in_fast)
fast.set_regimen(100,8)
fast.run_all(metrics_ref)


In [ ]:
#Visualize Conc-Time and PK Metrics 

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
plot_pk_curves(fast, sim_ref, ax1, strategies=['unadj','freq_opt'], xlim=(16, 18), ylim=(0, 600))
plot_metrics_bars(fast, metrics_ref, ax2, strategies=['unadj','freq_opt'])
plt.tight_layout()
plt.show()

base = (
    'Lorlatinib_Human_PKProject_30Jan2025/Hybrid_Model/Figures/'
)


# Save each panel individually
fig.canvas.draw()
panels = {
    'cin10_curves': ax1, 'cin10_bars': ax2
}
for name, ax in panels.items():
    extent = ax.get_tightbbox(fig.canvas.get_renderer()) \
               .transformed(fig.dpi_scale_trans.inverted())
   # fig.savefig(base + f'{name}.pdf', bbox_inches=extent, dpi=300)

In [ ]:
print_metrics(fast, metrics_ref)

In [ ]:
# ── Slow metabolizer ──────────────────────────────────────────────────────────
c_in_slow = 0.4
slow = Patient(c_in=c_in_slow)
slow.set_regimen(100,8)
slow.run_all(metrics_ref)


In [ ]:
#Visualize Conc-Time and PK Metrics 

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
plot_pk_curves(slow, sim_ref, ax1, strategies=['unadj','dose_opt'], xlim=(16, 18), ylim=(0, 1000))
plot_metrics_bars(slow, metrics_ref, ax2, strategies=['unadj','dose_opt'])
plt.tight_layout()
plt.show()

#Save entire figrue 
base = (
    'Lorlatinib_Human_PKProject_30Jan2025/Hybrid_Model/Figures/'
)


# Save each panel individually
fig.canvas.draw()
panels = {
    'cin0.4_Curves': ax1, 'cin0.4_Bars': ax2
}
for name, ax in panels.items():
    extent = ax.get_tightbbox(fig.canvas.get_renderer()) \
               .transformed(fig.dpi_scale_trans.inverted())
    #fig.savefig(base + f'{name}.pdf', bbox_inches=extent, dpi=300)

In [ ]:
print_metrics(slow, metrics_ref)


## 8. PK Metrics vs c_in 

In [ ]:
#Visualize entire c_in range

print("Computing steady-state PK metrics across c_in range 0.05 → 10...")
c_in_sweep = np.concatenate([np.linspace(0.05, 1.0, 30), np.linspace(1.0, 10.0, 40)[1:]])

auc_sweep, cmax_sweep, cmin_sweep = [], [], []
for i, c in enumerate(c_in_sweep):
    sim = HybridModel(c_in=c).simulate_multiple_doses(100, 18, 24)
    m   = calculate_final_metrics(sim, dosing_interval=24)
    auc_sweep.append(m['AUC_f'])
    cmax_sweep.append(m['Cmax_f'])
    cmin_sweep.append(m['Cmin_f'])
    if (i+1) % 20 == 0:
        print(f"  {i+1}/{len(c_in_sweep)}  c_in={c:.2f}  AUC={m['AUC_f']:.0f}")

c_in_sweep = np.array(c_in_sweep)
auc_sweep  = np.array(auc_sweep)
cmax_sweep = np.array(cmax_sweep)
cmin_sweep = np.array(cmin_sweep)
print(f"\u2713 Sweep complete ({len(c_in_sweep)} points)")

ref_idx    = np.argmin(np.abs(c_in_sweep - 1.0))
auc_ref_v  = auc_sweep[ref_idx]
cmax_ref_v = cmax_sweep[ref_idx]
cmin_ref_v = cmin_sweep[ref_idx]

COL_AUC  = '#212121'
COL_CMAX = '#C62828'
COL_CMIN = '#7B2D8B'
PHENO = [(0.4, 'Low',    'green'),
         (1.00, 'Average', 'blue'),
         (2.5, 'High',   '#ED7D31')]

from matplotlib.lines import Line2D

fig_cin, ax = plt.subplots(figsize=(13, 8))

l1, = ax.plot(c_in_sweep, auc_sweep/auc_ref_v*100,   '-', color=COL_AUC,  lw=2.5, zorder=3)
l2, = ax.plot(c_in_sweep, cmax_sweep/cmax_ref_v*100, '-', color=COL_CMAX, lw=2.0, zorder=3)
l3, = ax.plot(c_in_sweep, cmin_sweep/cmin_ref_v*100, '-', color=COL_CMIN, lw=2.0, zorder=3)

for c_val, name, col in PHENO:
    idx = np.argmin(np.abs(c_in_sweep - c_val))
    ax.axvline(c_in_sweep[idx], color=col, ls=':', lw=1.5, alpha=1, zorder=2)
    ax.scatter(c_in_sweep[idx], auc_sweep[idx]/auc_ref_v*100,   color=col, s=75, zorder=5)
    ax.scatter(c_in_sweep[idx], cmax_sweep[idx]/cmax_ref_v*100, color=col, s=65, zorder=5, marker='^'
)
    ax.scatter(c_in_sweep[idx], cmin_sweep[idx]/cmin_ref_v*100, color=col, s=65, zorder=5, marker='s')
    ax.text(c_val * 1, 0.96, f'{name}\nc$_{{in}}$={c_val}',
             ha='left', va='top', fontsize=15, color=col,
             transform=ax.get_xaxis_transform())

ax.set_yscale('log')
ax.set_xscale('linear')
ax.set_xlim(0.05, 10)
ax.set_xlabel('c$_{in}$ Value', fontsize=18)
ax.set_ylabel('% SOC', fontsize=18)
ax.grid(True, alpha=0.22, which='major', zorder=0)
ax.legend(handles=[l1, l2, l3],
           labels=['AUC (% SOC)', 'C$_{max}$ (% SOC)', 'C$_{min}$ (% SOC)'],
           fontsize=18, loc='upper right', framealpha=0.9)

plt.tight_layout()
plt.show()

out_cin = ('/Users/80031987/Desktop/Basanta_Marusyk_Lab_2026/'
           'Lorlatinib_Human_PKProject_30Jan2025/Hybrid_Model/Figures/'
           'Fig1B.pdf')
#fig_cin.savefig(out_cin, dpi=300, bbox_inches='tight')
print(f'\u2713 Saved: {out_cin}')


In [ ]:
# Zoomed view: visualize c_in range 0.05 – 1.0

mask = c_in_sweep <= 1.0
c_in_zoom   = c_in_sweep[mask]
auc_zoom    = auc_sweep[mask]
cmax_zoom   = cmax_sweep[mask]
cmin_zoom   = cmin_sweep[mask]

COL_AUC  = '#212121'
COL_CMAX = '#C62828'
COL_CMIN = '#7B2D8B'
PHENO = [(0.27, 'Slow',    'green'),
         (1.00, 'Average', 'blue')]

from matplotlib.lines import Line2D

fig_zoom, ax = plt.subplots(figsize=(13, 8))

l1z, = ax.plot(c_in_zoom, auc_zoom/auc_ref_v*100,   '-', color=COL_AUC,  lw=2.5, zorder=3)
l2z, = ax.plot(c_in_zoom, cmax_zoom/cmax_ref_v*100, '-', color=COL_CMAX, lw=2.0, zorder=3)
l3z, = ax.plot(c_in_zoom, cmin_zoom/cmin_ref_v*100, '-', color=COL_CMIN, lw=2.0, zorder=3)

for c_val, name, col in PHENO:
    if c_val > 1.0:
        continue
    idx = np.argmin(np.abs(c_in_zoom - c_val))
    ax.axvline(c_in_zoom[idx], color=col, ls=':', lw=1.5, alpha=0.65, zorder=2)
    ax.scatter(c_in_zoom[idx], auc_zoom[idx]/auc_ref_v*100,   color=col, s=75, zorder=5)
    ax.scatter(c_in_zoom[idx], cmax_zoom[idx]/cmax_ref_v*100, color=col, s=65, zorder=5, marker='^'
)
    ax.scatter(c_in_zoom[idx], cmin_zoom[idx]/cmin_ref_v*100, color=col, s=65, zorder=5, marker='s')
    ax.text(c_val*1.05, 0.96, f'{name}\nc$_{{in}}$={c_val}',
              ha='left', va='top', fontsize=18, color=col, fontweight='bold',
              transform=ax.get_xaxis_transform())

ax.set_yscale('log')
ax.set_xscale('linear')
ax.set_xlim(0.05, 1.0)
ax.set_xlabel('c$_{in}$ Value', fontsize=18)
ax.set_ylabel('% SOC', fontsize=18)
ax.grid(True, alpha=0.22, which='major', zorder=0)
ax.legend(handles=[l1z, l2z, l3z],
            labels=['AUC (% SOC)', 'C$_{max}$ (% SOC)', 'C$_{min}$ (% SOC)'],
            fontsize=18, loc='upper right', framealpha=0.9)

plt.tight_layout()
plt.show()

out_zoom = ('/Users/80031987/Desktop/Basanta_Marusyk_Lab_2026/'
            'Lorlatinib_Human_PKProject_30Jan2025/Hybrid_Model/Figures/'
            'Fig1_Supp1A.pdf')
#fig_zoom.savefig(out_zoom, dpi=300, bbox_inches='tight')
print(f'\u2713 Saved: {out_zoom}')


In [ ]:
# Zoomed view: visualize c_in range 1.0-10.0 

mask = c_in_sweep >= 1.0
c_in_zoom   = c_in_sweep[mask]
auc_zoom    = auc_sweep[mask]
cmax_zoom   = cmax_sweep[mask]
cmin_zoom   = cmin_sweep[mask]

COL_AUC  = '#212121'
COL_CMAX = '#C62828'
COL_CMIN = '#7B2D8B'
PHENO = [(0.27, 'Slow',    'green'),
         (1.00, 'Average', 'blue'),
         (2.52, 'Fast',    '#ED7D31')]

from matplotlib.lines import Line2D

fig_zoom, ax = plt.subplots(figsize=(13, 8))

l1z, = ax.plot(c_in_zoom, auc_zoom/auc_ref_v*100,   '-', color=COL_AUC,  lw=2.5, zorder=3)
l2z, = ax.plot(c_in_zoom, cmax_zoom/cmax_ref_v*100, '-', color=COL_CMAX, lw=2.0, zorder=3)
l3z, = ax.plot(c_in_zoom, cmin_zoom/cmin_ref_v*100, '-', color=COL_CMIN, lw=2.0, zorder=3)

for c_val, name, col in PHENO:
    if c_val < 1.0:
        continue
    idx = np.argmin(np.abs(c_in_zoom - c_val))
    ax.axvline(c_in_zoom[idx], color=col, ls=':', lw=1.5, alpha=0.65, zorder=2)
    ax.scatter(c_in_zoom[idx], auc_zoom[idx]/auc_ref_v*100,   color=col, s=75, zorder=5)
    ax.scatter(c_in_zoom[idx], cmax_zoom[idx]/cmax_ref_v*100, color=col, s=65, zorder=5, marker='^'
)
    ax.scatter(c_in_zoom[idx], cmin_zoom[idx]/cmin_ref_v*100, color=col, s=65, zorder=5, marker='s')
    y_pos = 0.86 if name == 'Fast' else 0.75
    ax.text(c_val*1.09, y_pos, f'{name}\nc$_{{in}}$={c_val}',
              ha='left', va='top', fontsize=18, color=col, fontweight='bold',
              transform=ax.get_xaxis_transform())

ax.set_yscale('log')
ax.set_xscale('linear')
ax.set_xlim(1.0, 10.0)
ax.set_xlabel('c$_{in}$', fontsize=18)
ax.set_ylabel('% SOC', fontsize=18)
ax.grid(True, alpha=0.22, which='major', zorder=0)
ax.legend(handles=[l1z, l2z, l3z],
            labels=['AUC (% SOC)', 'C$_{max}$ (% SOC)', 'C$_{min}$ (% SOC)'],
            fontsize=18, loc='lower left', framealpha=0.9)

plt.tight_layout()
plt.show()

out_zoom = ('/Users/80031987/Desktop/Basanta_Marusyk_Lab_2026/'
            'Lorlatinib_Human_PKProject_30Jan2025/Hybrid_Model/Figures/'
            'Fig1_Supp1B.pdf')
#fig_zoom.savefig(out_zoom, dpi=300, bbox_inches='tight')
print(f'\u2713 Saved: {out_zoom}')


## 9. Solution Set Visualization

Plot AUC vs Cmax across a dose range of 10–300 mg QD for each metabolizer phenotype.
Each curve traces the achievable (AUC, Cmax) space as dose increases.
The optimal dose (minimizing deviation from the reference) is marked with a star.

In [ ]:
# Plot PK Metrics vs Dose for 3 Metabolizer Phenotypes
# Each subplot is a PK Metric

dose_sweep_v2 = np.linspace(0, 1250, 60)

PHENO_SOL2 = [
    (0.4, 'Slow', 'green', 40),
    (1.00, 'Average', 'blue', 100.0),
    (2.5, 'Fast','#ED7D31', 250),
]

results = {}
for c_in_val, label, col, opt_dose in PHENO_SOL2:
    aucs, cmaxs, cmins = [], [], []
    for d in dose_sweep_v2:
        sim = HybridModel(c_in=c_in_val).simulate_multiple_doses(d, 18, 24)
        m   = calculate_final_metrics(sim, dosing_interval=24)
        aucs.append(m['AUC_f'])
        cmaxs.append(m['Cmax_f'])
        cmins.append(m['Cmin_f'])
    results[label] = dict(col=col, opt_dose=opt_dose,
                          aucs=np.array(aucs),
                          cmaxs=np.array(cmaxs),
                          cmins=np.array(cmins))
    print(f'  ✓ {label}')

ref_auc  = metrics_ref['AUC_f']
ref_cmax = metrics_ref['Cmax_f']
ref_cmin = metrics_ref['Cmin_f']

fig_dose, axes_dose = plt.subplots(1, 3, figsize=(22, 8))
plt.rcParams['axes.titlepad'] = 12

metrics_plot = [
    ('aucs',  ref_auc,  'AUC (% of SOC)',       (10, 300)),
    ('cmaxs', ref_cmax, 'C$_{max}$ (% of SOC)', (10, 300)),
    ('cmins', ref_cmin, 'C$_{min}$ (% of SOC)', (10, 300)),
]

for ax, (key, ref_val, ylabel, xlims) in zip(axes_dose, metrics_plot):
    for label, data in results.items():
        pct = data[key] / ref_val * 100
        ax.plot(dose_sweep_v2, pct, '-', color=data['col'],
                lw=2.0, alpha=0.85, label=label)
        opt_idx = np.argmin(np.abs(dose_sweep_v2 - data['opt_dose']))
        ax.scatter(dose_sweep_v2[opt_idx], pct[opt_idx],
                   color=data['col'], s=150, zorder=6, marker='*',
                   edgecolors='black', linewidths=0.6)

    ax.axhline(100, color='black', ls='--', lw=1.5, alpha=0.7, label='SOC (100%)')
    ax.set_xlabel('Dose (mg)', fontsize=18)
    ax.set_ylabel(ylabel, fontsize=18)
    ax.set_xlim(*xlims)
    ax.legend(loc='upper left', bbox_to_anchor=(1.0, 1.0), fontsize=18, frameon=False)

plt.tight_layout()
plt.show()

out_dose = ('/Users/80031987/Desktop/Basanta_Marusyk_Lab_2026/'
            'Lorlatinib_Human_PKProject_30Jan2025/Hybrid_Model/Figures/'
            'Fig4A.pdf')
#fig_dose.savefig(out_dose, dpi=300, bbox_inches='tight')
print(f'✓ Saved: {out_dose}')

In [ ]:
# Plot PK Metrics vs Dose for 3 Metabolizer Phenotypes
#Each Subplot is a metabolizer 
fig_d_sub, axes_d_sub = plt.subplots(1, 3, figsize=(22, 8))

ref_auc  = metrics_ref['AUC_f']
ref_cmax = metrics_ref['Cmax_f']
ref_cmin = metrics_ref['Cmin_f']

xlims_by_label = {
    'Slow':    (0, 80),
    'Average': (10, 300),
    'Fast':    (0, 800),
}

ylims_dose = {
    'Slow':          (0, 250),   
    'Average_left':  (0, 15000),   
    'Average_right': (0, 1500),   
    'Fast': (0, 300),   
}

for ax, (label, data) in zip(axes_d_sub, results.items()):
    if label == 'Average':
        ax2 = ax.twinx()
        ax.plot( dose_sweep_v2, data['aucs'],  '-', color=COL_AUC,  lw=2.0, alpha=0.9)
        ax2.plot(dose_sweep_v2, data['cmaxs'], '-', color=COL_CMAX, lw=1.8, alpha=0.9)
        ax2.plot(dose_sweep_v2, data['cmins'], '-', color=COL_CMIN, lw=1.8, alpha=0.9)

        for arr, tgt, ax_t, col_m in [
            (data['aucs'],  ref_auc,  ax,  COL_AUC),
            (data['cmaxs'], ref_cmax, ax2, COL_CMAX),
            (data['cmins'], ref_cmin, ax2, COL_CMIN),
        ]:
            if arr.min() <= tgt <= arr.max():
                idx = np.argmin(np.abs(arr - tgt))
                ax_t.scatter(dose_sweep_v2[idx], arr[idx],
                             color=col_m, s=200, zorder=7, marker='*',
                             edgecolors='black', linewidths=0.6)

        ax.set_yscale('linear')
        ax2.set_yscale('linear')
        ax.set_ylabel('AUC (ng·h/mL)', fontsize=18, color=COL_AUC)
        ax2.set_ylabel('C$_{max}$, C$_{min}$ (ng/mL)', fontsize=18, color='#212121')
        ax.tick_params(axis='y', labelcolor=COL_AUC)
        handles_d = [
            Line2D([0],[0], color=COL_AUC,  lw=2.0, ls='-', label='AUC'),
            Line2D([0],[0], color=COL_CMAX, lw=1.8, ls='-', label='C$_{max}$'),
            Line2D([0],[0], color=COL_CMIN, lw=1.8, ls='-', label='C$_{min}$'),
            Line2D([0],[0], color='gray', marker='*', markersize=10, ls='none',
                   markeredgecolor='black', label='SOC Metrics'),
        ]
    else:
        auc_pct  = data['aucs']  / ref_auc  * 100
        cmax_pct = data['cmaxs'] / ref_cmax * 100
        cmin_pct = data['cmins'] / ref_cmin * 100

        ax.plot(dose_sweep_v2, auc_pct,  '-', color=COL_AUC,  lw=2.0, alpha=0.9)
        ax.plot(dose_sweep_v2, cmax_pct, '-', color=COL_CMAX, lw=1.8, alpha=0.9)
        ax.plot(dose_sweep_v2, cmin_pct, '-', color=COL_CMIN, lw=1.8, alpha=0.9)

        ax.axhline(100, color='black', ls='--', lw=1.2, alpha=0.5)

        if auc_pct.min() <= 100 <= auc_pct.max():
            target_dose = np.interp(100, auc_pct, dose_sweep_v2)
            ax.axvline(target_dose, color='black', ls='--', lw=1.5, alpha=0.7)

        ax.set_ylabel('% of SOC', fontsize=18)
        handles_d = [
            Line2D([0],[0], color=COL_AUC,  lw=2.0, ls='-', label='AUC'),
            Line2D([0],[0], color=COL_CMAX, lw=1.8, ls='-', label='C$_{max}$'),
            Line2D([0],[0], color=COL_CMIN, lw=1.8, ls='-', label='C$_{min}$'),
            Line2D([0],[0], color='black',  lw=1.5, ls='--', alpha=0.7,
                   label='Target AUC dose'),
        ]

    ax.set_title(f'{label}', fontsize=18, fontweight='bold')
    ax.set_xlabel('Dose (mg)', fontsize=18)
    ax.set_xlim(*xlims_by_label[label])
    if label == 'Average':
        ax.set_ylim(*ylims_dose['Average_left'])
        ax2.set_ylim(*ylims_dose['Average_right'])
    else:
        ax.set_ylim(*ylims_dose[label])
    ax.legend(handles=handles_d, fontsize=12, loc='upper left', framealpha=0.9)

plt.tight_layout()
out_d_sub = ('/Users/80031987/Desktop/Basanta_Marusyk_Lab_2026/'
             'Lorlatinib_Human_PKProject_30Jan2025/Hybrid_Model/Figures/'
             'Fig2A.pdf')
#fig_d_sub.savefig(out_d_sub, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Save each metabolizer subplot individually

fig_d_sub.canvas.draw()
renderer_d = fig_d_sub.canvas.get_renderer()

for ax, label in zip(axes_d_sub, results.keys()):
    out_d_ind = '/Users/80031987/Desktop/Basanta_Marusyk_Lab_2026/Lorlatinib_Human_PKProject_30Jan2025/Hybrid_Model/Figures/' + f'Fig2A_{label}.pdf'
    extent = ax.get_tightbbox(renderer_d).transformed(
        fig_d_sub.dpi_scale_trans.inverted()
    )
    #fig_d_sub.savefig(out_d_ind, bbox_inches=extent, dpi=300)
    print(f'\u2713 Saved: {out_d_ind}')


In [ ]:
# Plot PK Metrics vs Dosing Frequency for 3 Metabolizer Phenotypes
# Each subplot is a PK Metric

interval_sweep2 = np.geomspace(1, 96, 120)  

PHENO_FREQ = [
    (0.4, 'Slow', 'green', 60),
    (1.00, 'Average', 'blue', 24.0),
    (2.5, 'Fast', '#ED7D31', 9.6),
]

results_freq = {}
for c_in_val, label, col, opt_interval in PHENO_FREQ:
    cavgs, cmaxs, cmins = [], [], []
    for tau in interval_sweep2:
        n = max(20, int(500 / tau))
        sim = HybridModel(c_in=c_in_val).simulate_multiple_doses(100, n, tau)
        m   = calculate_final_metrics(sim, dosing_interval=tau)
        cavgs.append(m['Cavg_f'])
        cmaxs.append(m['Cmax_f'])
        cmins.append(m['Cmin_f'])
    from scipy.ndimage import gaussian_filter1d
    results_freq[label] = dict(col=col, opt_interval=opt_interval,
                               cavgs=gaussian_filter1d(np.array(cavgs), sigma=2),
                               cmaxs=gaussian_filter1d(np.array(cmaxs), sigma=2),
                               cmins=gaussian_filter1d(np.array(cmins), sigma=2))
    print(f'  ✓ {label}')

ref_cavg = metrics_ref['Cavg_f']
ref_cmax = metrics_ref['Cmax_f']
ref_cmin = metrics_ref['Cmin_f']

fig_fq_imp, axes_fq = plt.subplots(1, 3, figsize=(22, 8))
plt.rcParams['axes.titlepad'] = 12

metrics_fq = [
    ('cavgs', ref_cavg, 'AUC/$\\tau$ (% of SOC)'),
    ('cmaxs', ref_cmax, 'C$_{max}$ (% of SOC)'),
    ('cmins', ref_cmin, 'C$_{min}$ (% of SOC)'),
]

for ax, (key, ref_val, ylabel) in zip(axes_fq, metrics_fq):
    for label, data in results_freq.items():
        pct = data[key] / ref_val * 100
        ax.plot(interval_sweep2, pct, '-', color=data['col'],
                lw=2.0, alpha=0.85, label=label)
        opt_idx = np.argmin(np.abs(interval_sweep2 - data['opt_interval']))
        ax.scatter(interval_sweep2[opt_idx], pct[opt_idx],
                   color=data['col'], s=150, zorder=6, marker='*',
                   edgecolors='black', linewidths=0.6)

    ax.axhline(100, color='black', ls='--', lw=1.5, alpha=0.7, label='SOC (100%)')
    ax.set_xlabel('Dosing Interval (h)', fontsize=18)
    ax.set_ylabel(ylabel, fontsize=18)
    ax.set_xlim(0, 96)
    ax.legend(loc='upper left', bbox_to_anchor=(1.0, 1.0), fontsize=18, frameon=False)

plt.tight_layout()
plt.show()

out_fq_imp = ('/Users/80031987/Desktop/Basanta_Marusyk_Lab_2026/'
              'Lorlatinib_Human_PKProject_30Jan2025/Hybrid_Model/Figures/'
              'Fig4B.pdf')
#fig_fq_imp.savefig(out_fq_imp, dpi=300, bbox_inches='tight')
print(f'✓ Saved: {out_fq_imp}')

In [ ]:
# Plot PK Metrics vs Dosing Frequency for 3 Metabolizer Phenotypes
# Each subplot is a metabolizer 

fig_f_sub, axes_f_sub = plt.subplots(1, 3, figsize=(22, 8))

ref_cavg = metrics_ref['Cavg_f']
ref_cmax = metrics_ref['Cmax_f']
ref_cmin = metrics_ref['Cmin_f']

xlims_by_label_freq = {
    'Slow':    (40, 80),
    'Average': (1, 48),
    'Fast': (1, 24),
}

ylims_freq = {
    'Slow':          (80, 120),   
    'Average_left':  (0, 1000),   
    'Average_right': (0, 1000),   
    'Fast':    (0, 200),   
}

for ax, (label, data) in zip(axes_f_sub, results_freq.items()):
    if label == 'Average':
        ax2 = ax.twinx()
        ax.plot( interval_sweep2, data['cavgs'], '-', color=COL_AUC,  lw=2.0, alpha=0.9)
        ax2.plot(interval_sweep2, data['cmaxs'], '-', color=COL_CMAX, lw=1.8, alpha=0.9)
        ax2.plot(interval_sweep2, data['cmins'], '-', color=COL_CMIN, lw=1.8, alpha=0.9)

        for arr, tgt, ax_t, col_m in [
            (data['cavgs'], ref_cavg, ax,  COL_AUC),
            (data['cmaxs'], ref_cmax, ax2, COL_CMAX),
            (data['cmins'], ref_cmin, ax2, COL_CMIN),
        ]:
            if arr.min() <= tgt <= arr.max():
                idx = np.argmin(np.abs(arr - tgt))
                ax_t.scatter(interval_sweep2[idx], arr[idx],
                             color=col_m, s=200, zorder=7, marker='*',
                             edgecolors='black', linewidths=0.6)

        ax.set_yscale('linear')
        ax2.set_yscale('linear')
        ax.set_ylabel('AUC/$\\tau$ (ng/mL)', fontsize=18, color=COL_AUC)
        ax2.set_ylabel('C$_{max}$, C$_{min}$ (ng/mL)', fontsize=18, color='#212121')
        ax.tick_params(axis='y', labelcolor=COL_AUC)
        handles_f = [
            Line2D([0],[0], color=COL_AUC,  lw=2.0, ls='-', label='AUC/$\\tau$'),
            Line2D([0],[0], color=COL_CMAX, lw=1.8, ls='-', label='C$_{max}$'),
            Line2D([0],[0], color=COL_CMIN, lw=1.8, ls='-', label='C$_{min}$'),
            Line2D([0],[0], color='gray', marker='*', markersize=10, ls='none',
                   markeredgecolor='black', label='SOC Metrics'),
        ]
    else:

        cavg_pct = data['cavgs'] / ref_cavg * 100
        cmax_pct = data['cmaxs'] / ref_cmax * 100
        cmin_pct = data['cmins'] / ref_cmin * 100

        ax.plot(interval_sweep2, cavg_pct, '-', color=COL_AUC,  lw=2.0, alpha=0.9)
        ax.plot(interval_sweep2, cmax_pct, '-', color=COL_CMAX, lw=1.8, alpha=0.9)
        ax.plot(interval_sweep2, cmin_pct, '-', color=COL_CMIN, lw=1.8, alpha=0.9)

        ax.axhline(100, color='black', ls='--', lw=1.2, alpha=0.5)

        if cavg_pct.min() <= 100 <= cavg_pct.max():
            target_interval = np.interp(100, cavg_pct[::-1], interval_sweep2[::-1])
            ax.axvline(target_interval, color='black', ls='--', lw=1.5, alpha=0.7)

        ax.set_ylabel('% of SOC', fontsize=18)
        handles_f = [
            Line2D([0],[0], color=COL_AUC,  lw=2.0, ls='-', label='AUC/$\\tau$'),
            Line2D([0],[0], color=COL_CMAX, lw=1.8, ls='-', label='C$_{max}$'),
            Line2D([0],[0], color=COL_CMIN, lw=1.8, ls='-', label='C$_{min}$'),
            Line2D([0],[0], color='black',  lw=1.5, ls='--', alpha=0.7,
                   label='Target Cavg interval'),
        ]

    ax.set_title(f'{label}', fontsize=18, fontweight='bold')
    ax.set_xlabel('Dosing Interval (h)', fontsize=18)
    ax.set_xlim(*xlims_by_label_freq[label])
    if label == 'Average':
        ax.set_ylim(*ylims_freq['Average_left'])
        ax2.set_ylim(*ylims_freq['Average_right'])
    else:
        ax.set_ylim(*ylims_freq[label])
    ax.legend(handles=handles_f, fontsize=12, loc='upper right', framealpha=0.9)

plt.tight_layout()
out_f_sub = ('/Users/80031987/Desktop/Basanta_Marusyk_Lab_2026/'
             'Lorlatinib_Human_PKProject_30Jan2025/Hybrid_Model/Figures/'
             'Fig2B.pdf')
#fig_f_sub.savefig(out_f_sub, dpi=300, bbox_inches='tight')
print(f'✓ Saved: {out_f_sub}')
plt.show()

In [ ]:
# Save each metabolizer subplot individually

fig_f_sub.canvas.draw()
renderer_f = fig_f_sub.canvas.get_renderer()

for ax, label in zip(axes_f_sub, results_freq.keys()):
    out_f_ind = '/Users/80031987/Desktop/Basanta_Marusyk_Lab_2026/Lorlatinib_Human_PKProject_30Jan2025/Hybrid_Model/Figures/' + f'Fig2B_{label}.pdf'
    extent = ax.get_tightbbox(renderer_f).transformed(
        fig_f_sub.dpi_scale_trans.inverted()
    )
    #fig_f_sub.savefig(out_f_ind, bbox_inches=extent, dpi=300)
    print(f'\u2713 Saved: {out_f_ind}')
